# Day 2-04｜多個 Detection Box 投影到 BEV

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：把多個 player box 的 bottom-center 投影到 BEV。這是 Day 1 Homography + Day 2 Detection 的第一次組合。

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
from src.cv_utils import (
    read_image_rgb,
    load_json,
    draw_boxes,
    draw_points,
    show_image,
    side_by_side,
    bottom_center,
    save_image_rgb,
)
from src.geometry_utils import compute_homography, project_points

image = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png")
bev = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_bev_court.png")
homo = load_json(COURSE_ROOT / "assets" / "samples" / "sample_homography_points.json")
H = compute_homography(homo["camera_points"], homo["bev_points"])

data = load_json(COURSE_ROOT / "assets" / "samples" / "sample_detections_frame0.json")
players = [
    d
    for d in data["detections"]
    if d["class_name"] == "player" and d["confidence"] >= 0.75
]

boxes = [d["bbox_xyxy"] for d in players]
feet = [bottom_center(b) for b in boxes]
bev_feet = project_points(feet, H)

print("players:", len(players))

In [ ]:
labels = [f"p{i}" for i in range(len(players))]
frame_vis = draw_boxes(image, boxes, labels)
frame_vis = draw_points(frame_vis, feet, labels)
bev_vis = draw_points(bev, bev_feet, labels)
combo = side_by_side(frame_vis, bev_vis)
show_image(combo, "multi-player BBOX-to-BEV")

save_path = COURSE_ROOT / "assets" / "results" / "d2_04_bbox_to_bev_integration.png"
save_image_rgb(save_path, combo)
print("saved:", save_path)

這裡先不管 track ID。下一天才會處理「同一個人跨 frame 如何保持同一個 ID」。